# Zomato Sales Analysis — Data Cleaning & EDA
**Student:** Subham Srinibas Behera  
**Project:** Zomato Sales Analysis — Business Intelligence Project  
**Tool:** Python (Pandas, Matplotlib, Seaborn)  
**Dataset:** Zomato Orders Dataset (India)


In [ ]:
# ── 1. Import Libraries ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully ✅')

In [ ]:
# ── 2. Load Dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('accident_records.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# ── 3. Basic Info & Data Types ───────────────────────────────────────────────
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===')
print(f'Duplicates found: {df.duplicated().sum()}')

In [ ]:
# ── 4. Data Cleaning ─────────────────────────────────────────────────────────
# Parse date column
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Drop duplicates if any
df.drop_duplicates(inplace=True)

# Strip whitespace from object columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

# Validate: Final_Amount = Order_Value - Discount_Applied
df['Calc_Final'] = df['Order_Value'] - df['Discount_Applied']
mismatch = df[df['Calc_Final'] != df['Final_Amount']]
print(f'Amount mismatches: {len(mismatch)}')
df.drop(columns=['Calc_Final'], inplace=True)

print('\n✅ Data cleaning complete!')
print(f'Clean dataset shape: {df.shape}')

In [ ]:
# ── 5. Descriptive Statistics ────────────────────────────────────────────────
print('=== Descriptive Statistics ===')
df[['Order_Value', 'Final_Amount', 'Delivery_Time_mins', 'Rating', 'Discount_Applied']].describe().round(2)

In [ ]:
# ── 6. Revenue by City ───────────────────────────────────────────────────────
city_rev = df.groupby('City')['Final_Amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(city_rev.index, city_rev.values, color=sns.color_palette('viridis', len(city_rev)))
ax.set_title('Total Revenue by City', fontsize=14, fontweight='bold')
ax.set_xlabel('City')
ax.set_ylabel('Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'₹{bar.get_height():,.0f}', ha='center', fontsize=8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('dashboard_page-0001.jpg', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as dashboard_page-0001.jpg')

In [ ]:
# ── 7. Orders by Cuisine Type ────────────────────────────────────────────────
cuisine_cnt = df['Cuisine_Type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
axes[0].pie(cuisine_cnt.values, labels=cuisine_cnt.index, autopct='%1.1f%%',
            startangle=140, colors=sns.color_palette('Set3', len(cuisine_cnt)))
axes[0].set_title('Order Distribution by Cuisine', fontsize=13, fontweight='bold')

# Bar chart — avg rating by cuisine
avg_rating = df.groupby('Cuisine_Type')['Rating'].mean().sort_values(ascending=True)
axes[1].barh(avg_rating.index, avg_rating.values,
             color=sns.color_palette('RdYlGn', len(avg_rating)))
axes[1].set_title('Avg Customer Rating by Cuisine', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average Rating')
axes[1].axvline(avg_rating.mean(), color='red', linestyle='--', label='Mean Rating')
axes[1].legend()

plt.tight_layout()
plt.savefig('dashboard_page-0002.jpg', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as dashboard_page-0002.jpg')

In [ ]:
# ── 8. Payment Mode & Weekend Analysis ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Payment mode distribution
pay_cnt = df['Payment_Mode'].value_counts()
axes[0].pie(pay_cnt, labels=pay_cnt.index, autopct='%1.1f%%', startangle=90,
            colors=['#FF6B6B','#4ECDC4','#45B7D1'])
axes[0].set_title('Payment Mode Distribution', fontsize=13, fontweight='bold')

# Weekend vs Weekday avg order value
wk_avg = df.groupby('Is_Weekend')['Final_Amount'].mean()
wk_avg.index = ['Weekday', 'Weekend']
axes[1].bar(wk_avg.index, wk_avg.values, color=['#5C85D6','#FF8C42'], width=0.4)
axes[1].set_title('Avg Order Value: Weekend vs Weekday', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Avg Order Value (₹)')
for i, v in enumerate(wk_avg.values):
    axes[1].text(i, v + 5, f'₹{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print('Analysis complete!')

In [ ]:
# ── 9. Monthly Revenue Trend ─────────────────────────────────────────────────
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly = df.groupby('Month')['Final_Amount'].sum().reindex(
    [m for m in month_order if m in df['Month'].unique()])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly.index, monthly.values, marker='o', linewidth=2.5,
        color='#E63946', markersize=8)
ax.fill_between(range(len(monthly)), monthly.values, alpha=0.15, color='#E63946')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly.index, rotation=30)
ax.set_title('Monthly Revenue Trend (2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Export Summary CSVs ──────────────────────────────────────────────────
# Cuisine summary
cuisine_summary = df.groupby('Cuisine_Type').agg(
    Total_Orders=('Order_ID','count'),
    Total_Revenue=('Final_Amount','sum'),
    Avg_Order_Value=('Final_Amount','mean'),
    Avg_Rating=('Rating','mean'),
    Avg_Delivery_mins=('Delivery_Time_mins','mean')
).round(2).reset_index()
cuisine_summary.to_csv('cuisine_summary.csv', index=False)

# City summary
city_summary = df.groupby('City').agg(
    Total_Orders=('Order_ID','count'),
    Total_Revenue=('Final_Amount','sum'),
    Avg_Rating=('Rating','mean'),
    Avg_Delivery_mins=('Delivery_Time_mins','mean')
).round(2).reset_index()
city_summary.to_csv('city_sales_index.csv', index=False)

print('✅ All summary CSVs exported!')
print('\n=== Project Summary ===')
print(f'Total Orders Analysed : {len(df)}')
print(f'Total Revenue         : ₹{df["Final_Amount"].sum():,.0f}')
print(f'Avg Order Value       : ₹{df["Final_Amount"].mean():,.0f}')
print(f'Avg Customer Rating   : {df["Rating"].mean():.2f} / 5.0')
print(f'Avg Delivery Time     : {df["Delivery_Time_mins"].mean():.1f} minutes')
print(f'Cities Covered        : {df["City"].nunique()}')
print(f'Cuisines Tracked      : {df["Cuisine_Type"].nunique()}')